# One-sided Hoeffding Inequality For Prior Likelihood Mixing

The Prior Likelihood Mixing Theorem is states that
$$
  C_t = \left\{\theta \in \Theta: L_t(\theta) \leq \frac{1}{\delta} - \log \int \prod_{s=1}^t p_s(y_s \mid \nu) \; d \mu_0(\nu)\right\}
$$
is a $(1-\delta)$-confidence sequence.

The one-sided version of Hoeffding's inequality (https://cs229.stanford.edu/extra-notes/hoeffding.pdf) applies to settings where we have independent random variables $Z_1, \dots, Z_n$ such that for all $i \in [n]$, there exist $a, b \in \mathbb{R}$ for which $a \leq Z_i \leq $. 
Hoeffding's theorem states that for all $t \in (0, \infty)$
$$
  \Pr\left(\frac{1}{n} \sum_{i=1}^n (Z_i - E[Z_i]) \geq t\right) \leq \exp\left(- \frac{2nt^2}{(b - a)^2}\right)
$$

Since we assume that for all time steps $s \in \mathbb{N}$, all images $\nu \in [0, 1]^{r^2}$ and all data sequences $((x_s, y_s), s \in \mathbb{N})$
$$
  p_s(y_s \mid \nu) = \prod_{i=1}^r \mathrm{Pois}(y_{s, i}, I_0 \exp(-\mathcal{R}(\nu, x_s)))
$$
we have that
$$
  \prod_{s=1}^t p_s(y_s \mid \nu) \in [0, 1].
$$

Define for all $i \in \mathbb{N}$, $Z_i \coloneqq \prod_{s=1}^t p_s(y_s \mid \vartheta)$ and let $\vartheta \sim \mu_0$. 
Furthermore, assume that all $Z_1, \dots, Z_n$ are i.i.d. Then applying Hoeffding's inequality yields
$$
  \Pr\left(\frac{1}{n} \sum (Z_i - E Z_i) \geq t\right) \leq \exp\left(- \frac{2nt^2}{(b_i - a_i)^2}\right)
$$
The i.i.d. assumption implies
$$
  \Pr\left(\frac{1}{n} \sum (Z_i - E Z_i) \geq t\right) = \Pr\left(t + \frac{1}{n} \sum Z_i \leq E Z_1\right) \leq \exp\left(- \frac{2nt^2}{(b - a)^2}\right).
$$
Let 
$$
  \delta = \exp\left(- \frac{2nt^2}{(b - a)^2}\right).
$$
Choose error tolerance $t = 0.01$, error level $\delta \leq 0.025$. 
Then plugging $a, b$ and $t$ into the RHS yields
$$
  \log \delta = -0.0002n \leq \log 0.025 \iff n \geq 18444.397\dots
$$
So a sample size of $n = 18445$ suffices.

## Sequential Application of Hoeffding's Inequality

For each round $r=1,\dots,200$, define

$$
Z_i^{(r)} \coloneqq \prod_{s=1}^{t_r} p_s(y_s\mid \vartheta_i)\in[0,1],\qquad
\hat Z^{(r)} \coloneqq \frac{1}{n}\sum_{i=1}^n Z_i^{(r)}.
$$

By one-sided Hoeffding (with $a=0,b=1$), for any fixed $r$ and any deviation level $t>0$,

$$
\Pr\left( \mathbb{E}[Z_1^{(r)}] - \hat{Z}^{(r)} \geq t \right) \leq \exp(-2nt^2).
$$

Apply a union bound over the 200 rounds:

$$
\Pr\left( \exists r \in [200] : \mathbb{E}[Z_1^{(r)}] - \hat{Z}^{(r)} \geq t \right)
\leq 200 \exp(-2nt^2).
$$

To ensure this probability is at most $\delta = 0.025$, it suffices that

$$
200 \exp(-2nt^2) \leq \delta
\iff
n \geq \frac{1}{2t^{2}}\ln \left(\frac{200}{\delta}\right).
$$

With your choices $t=0.01$ and $\delta=0.025$,

$$
n \ge \frac{1}{2(0.01)^2} \ln \left(\frac{200}{0.025}\right)
= 5000\ln(8000) \approx 44935.98.
$$

Therefore, $n=44936$ samples suffice to guarantee, with probability at least $97.5\%$, that for all $r=1,\dots,200$,

$$
\mathbb{E}[Z_1^{(r)}] \leq \hat Z^{(r)} + 0.01.
$$


In [27]:
# Parameters from the write-up
import math
import pandas as pd

# Single-round parameters
a, b = 0.0, 1.0
t = 0.01           # deviation level (epsilon)
delta_single = 0.025   # error level for single round

# Multi-round parameters
R = 200                # number of rounds
delta_multi = 0.025    # overall error level across all rounds

def hoeffding_one_sided_n(a, b, t, delta):
    """
    Minimum n for one-sided Hoeffding bound:
    P(E[Z] - Zhat >= eps) <= delta,
    for Z in [a,b].
    
    n >= ( (b-a)^2 / (2 eps^2) ) * ln(1/delta)
    """
    width = b - a
    return (width**2) / (2 * t**2) * math.log(1/delta)

def hoeffding_union_over_rounds_n(a, b, eps, delta, rounds):
    """
    Minimum n for one-sided Hoeffding bound to hold simultaneously over `rounds` rounds
    via union bound:
    
    P( exists r in [rounds] : E[Z^(r)] - Zhat^(r) >= eps ) <= delta
    
    n >= ( (b-a)^2 / (2 eps^2) ) * ln(rounds / delta)
    """
    width = b - a
    return (width**2) / (2 * t**2) * math.log(rounds / delta)

# Compute results
n_single = hoeffding_one_sided_n(a, b, t, delta_single)
n_multi  = hoeffding_union_over_rounds_n(a, b, t, delta_multi, R)

# Collect detailed intermediate numbers
results = pd.DataFrame([
    {
        "Scenario": "Single round",
        "a": a, "b": b, "t": t, "delta": delta_single, "rounds": 1,
        "ln(1/delta)": math.log(1/delta_single),
        "1/(2*t^2)": 1/(2*t**2),
        "n (exact)": n_single,
        "n (ceil)": math.ceil(n_single),
    },
    {
        "Scenario": f"Uniform over {R} rounds",
        "a": a, "b": b, "t": t, "delta": delta_multi, "rounds": R,
        "ln(rounds/delta)": math.log(R/delta_multi),
        "1/(2*t^2)": 1/(2*t**2),
        "n (exact)": n_multi,
        "n (ceil)": math.ceil(n_multi),
    },
])


# Also print the key values explicitly
print("=== Single round ===")
print(f"t = {t}, delta = {delta_single}")
print(f"n >= ( (b-a)^2 / (2 t^2) ) * ln(1/delta)")
print(f"n >= {1/(2*t**2):.0f} * ln(1/{delta_single}) = {n_single:.2f}")
print(f"n_ceiling = {math.ceil(n_single)}")

print("\n=== Uniform over R rounds via union bound ===")
print(f"R = {R}, t = {t}, delta = {delta_multi}")
print(f"n >= ( (b-a)^2 / (2 t^2) ) * ln(R/delta)")
print(f"n >= {1/(2*t**2):.0f} * ln({R}/{delta_multi}) = {n_multi:.2f}")
print(f"n_ceiling = {math.ceil(n_multi)}")

=== Single round ===
t = 0.01, delta = 0.025
n >= ( (b-a)^2 / (2 t^2) ) * ln(1/delta)
n >= 5000 * ln(1/0.025) = 18444.40
n_ceiling = 18445

=== Uniform over R rounds via union bound ===
R = 200, t = 0.01, delta = 0.025
n >= ( (b-a)^2 / (2 t^2) ) * ln(R/delta)
n >= 5000 * ln(200/0.025) = 44935.98
n_ceiling = 44936


In [29]:
# Simulate Z_i directly via Beta distribution (no Radon/CT) and verify the uniform one-sided Hoeffding bound over 200 rounds.
# - You set how to generate Z ~ distribution on [0,1].
# - We simulate n i.i.d. samples per round, compute hat{Z}^{(r)}, and check the union-bound guarantee.

import math
import numpy as np
import pandas as pd

# -----------------------
# Parameters (edit these)
# -----------------------
R = 200         # number of rounds
n = 44936       # samples per round (e.g., ceiling from earlier)
# n = 100       # WRONG n to simulate higher failure probability
delta = 0.025   # overall error level
a, b = 0.0, 1.0 # bounds for Z
t = 0.01 # safety margin
n = math.ceil(hoeffding_union_over_rounds_n(a, b, t, delta, R))

# -----------------------
# Z generator on [0,1]
# -----------------------
rng = np.random.default_rng()

def sample_Z(size):
    alpha, beta_param = 2.0, 5.0
    return rng.beta(alpha, beta_param, size=size)

# -----------------------
# Simulation
# -----------------------
width = b - a

# Simulate rounds
hatZ = np.empty(R)
EZ_true = np.empty(R)  # unknown in practice; here only for simulation diagnostics

# If you know the true expectation for your chosen generator, set it here;
# for Beta(alpha,beta), mean = alpha/(alpha+beta). For uniform, 0.5. For Bernoulli(p), p.
# We'll infer it from the generator chosen above for diagnostics.
def true_mean_for_generator():
    # If using Beta(2,5):
    return 2.0 / (2.0 + 5.0)
    # If using Uniform[0,1]: return 0.5
    # If using Bernoulli(p): return p

EZ_guess = true_mean_for_generator()

for r in range(R):
    Z = sample_Z(n)
    hatZ[r] = Z.mean()
    EZ_true[r] = EZ_guess  # constant across rounds in this example

# Check violations: E[Z] - hatZ >= target_eps  (unknown E[Z] in practice; here we use EZ_true for diagnostic)
violations_target = (EZ_true >= hatZ + t).sum()

# Build per-round table
df = pd.DataFrame({
    "round": np.arange(1, R+1),
    "hatZ": hatZ,
    "diagnostic_EZ_true": EZ_true,            # diagnostic only
    "diagnostic_gap_to_true": EZ_true - hatZ, # diagnostic only
    "uniform_eps": np.full(R, t),
    "upper_bound_hatZ_plus_uniform_eps": hatZ + t,
})

# Print a compact summary
print("=== Parameters ===")
print(f"R={R}, n={n}, delta={delta}, [a,b]=[{a},{b}]")
print(f"t = {t}")
print(f"t (from n, R, delta) = {t:.6f}")

print("\n=== Diagnostics (useful since we know E[Z] here) ===")
print(f"Assumed true E[Z] = {EZ_guess:.6f}")
print(f"Max (E[Z] - hatZ) over rounds = {(EZ_true - hatZ).max():.6f}")
print(f"#violations of (E[Z] - hatZ >= target_eps) = {violations_target}")

=== Parameters ===
R=200, n=44936, delta=0.025, [a,b]=[0.0,1.0]
t = 0.01
t (from n, R, delta) = 0.010000

=== Diagnostics (useful since we know E[Z] here) ===
Assumed true E[Z] = 0.285714
Max (E[Z] - hatZ) over rounds = 0.002883
#violations of (E[Z] - hatZ >= target_eps) = 0
